In [1]:
from stage_discharge.read_write import read_hobo_log, read_vusitu_log, clean_hobo_log, clean_vusitu_log
from stage_discharge.build_stage import calc_diff_pressure, calc_depth
from stage_discharge.hobo_fetch import fetch_licor_data, tidy_licor_data
from stage_discharge.rating_curve import RatingCurve
import pandas as pd
import matplotlib.pyplot as plt


In [7]:
header_abs, data_abs, units_abs = read_vusitu_log(r"C:\Users\whe255\dev\stage_discharge\imports\tests\VuSitu_Log_2025-10-15_18-00-00_dolan_xing_tube_20251015_dolanXing.csv", pressure_kind='absolute')
data_abs = clean_vusitu_log(data_abs, units_abs)

header_baro, data_baro, units_baro = read_vusitu_log(r"C:\Users\whe255\dev\stage_discharge\imports\tests\VuSitu_Log_2025-10-15_18-00-00_Dolan_Baro_20251015_DolanBaro.csv", pressure_kind='barometric')
data_baro = clean_vusitu_log(data_baro, units_baro)

data, units = read_hobo_log(r"C:\Users\whe255\dev\stage_discharge\imports\tests\DEV300_20260914_export3-2026_09_14_14_16_02_UTC.csv")
data = clean_hobo_log(data, units)

C:\Users\whe255\dev\stage_discharge\src\stage_discharge\read_write.py:206: UserWarning: HOBO: unrecognized measurement 'line' -- passing it through un-aliased. Add it to standard_names if it's real.
  key = _apply_alias(key, sn.HOBO_ALIASES, "HOBO")


In [11]:
data

(                           diff_pressure_pa  abs_pressure_pa  temperature_c  \
 datetime                                                                      
 2026-04-22 16:15:00+00:00               NaN              NaN      20.944444   
 2026-04-22 16:20:00+00:00               NaN              NaN      20.944444   
 2026-04-22 16:25:00+00:00               NaN              NaN      20.944444   
 2026-04-22 16:30:00+00:00               NaN              NaN      20.944444   
 2026-04-22 16:35:00+00:00               NaN              NaN      21.000000   
 ...                                     ...              ...            ...   
 2026-09-11 19:15:00+00:00        347.495753     96665.182616      30.611111   
 2026-09-11 19:30:00+00:00        333.706239     96623.814074      30.722222   
 2026-09-11 19:45:00+00:00        335.774666     96599.682424      30.777778   
 2026-09-11 20:00:00+00:00        344.737850     96575.550775      30.888889   
 2026-09-11 20:15:00+00:00        325.43

In [3]:
vudolan = pd.merge_asof(
    data_abs.sort_index(), data_baro.sort_index(),
    left_index=True, right_index=True,
    direction="nearest",
    tolerance=pd.Timedelta("1s"),      # pair a reading only with a baro sample within 1 s
    suffixes=("_abs", "_baro"),
)

In [4]:
diff_log = calc_diff_pressure(vudolan['abs_pressure_psi'], vudolan['baro_pressure_psi'])
stage_log = calc_depth(diff_log,vudolan['temperature_c_abs'])

In [5]:
dolan_hobo = fetch_licor_data(serial_number=22475388, start_date="2026-07-01 12:00:00", end_date="2026-09-01 12:00:00", token = 'oJ5W88JkMtRTxYnTtMXGSal5v3iOwqD728WvKnulhIsHXjw3', report = True)
dolan_hobo, units = tidy_licor_data(dolan_hobo)

OK: Found: 29765 results., sensors = ['Absolute Pressure', 'Barometric Pressure', 'Diff Pressure', 'Temperature', 'Water Level']


In [6]:
dolan_hobo

sensor_measurement_type,abs_pressure_psi,baro_pressure_psi,diff_pressure_psi,temperature_f,water_level_feet
timestamp,,,,,
2026-07-01 12:00:00+00:00,14.113930,14.028486,0.085444,75.709608,0.609553
2026-07-01 12:15:00+00:00,14.110714,14.026455,0.084259,75.709608,0.607008
2026-07-01 12:30:00+00:00,14.111358,14.028195,0.083162,75.709608,0.604463
2026-07-01 12:45:00+00:00,14.111358,14.028921,0.082437,75.709608,0.602612
2026-07-01 13:00:00+00:00,14.114573,14.031966,0.082606,75.709608,0.603075
...,...,...,...,...,...
2026-09-01 11:00:00+00:00,14.100176,14.034432,0.065744,78.328309,0.564030
2026-09-01 11:15:00+00:00,14.102106,14.035881,0.066225,78.328309,0.565188
2026-09-01 11:30:00+00:00,14.101463,14.037914,0.063549,78.328309,0.558938


In [7]:
print(dolan_hobo.head())
plt.plot(dolan_hobo['water_level'][-10000:-1])

sensor_measurement_type    abs_pressure_psi  baro_pressure_psi  \
timestamp                                                        
2026-07-01 12:00:00+00:00         14.113930          14.028486   
2026-07-01 12:15:00+00:00         14.110714          14.026455   
2026-07-01 12:30:00+00:00         14.111358          14.028195   
2026-07-01 12:45:00+00:00         14.111358          14.028921   
2026-07-01 13:00:00+00:00         14.114573          14.031966   

sensor_measurement_type    diff_pressure_psi  temperature_f  water_level_feet  
timestamp                                                                      
2026-07-01 12:00:00+00:00           0.085444      75.709608          0.609553  
2026-07-01 12:15:00+00:00           0.084259      75.709608          0.607008  
2026-07-01 12:30:00+00:00           0.083162      75.709608          0.604463  
2026-07-01 12:45:00+00:00           0.082437      75.709608          0.602612  
2026-07-01 13:00:00+00:00           0.082606      75.7096

KeyError: 'water_level'